# Exercises XP: Prompt Engineering

**What you'll learn:**
- how to analyse and improve vague prompts;
- role prompting, constraint prompting and tone specification;
- how to structure multi-part prompts for specific use cases.

**How this notebook works:** every prompt is written in a Markdown/code cell **and executed** on an open instruction-tuned model (`Qwen/Qwen2.5-1.5B-Instruct`, free, runs on Colab CPU or GPU, no API key). That way the vague and improved prompts can be compared on real outputs. The same prompts can be pasted into ChatGPT; a stronger model will follow the constraints even more closely.

In [ ]:
!pip install -q "transformers<5" accelerate

In [ ]:
import os, re, textwrap, time
os.environ["USE_TF"] = "0"
import torch
from transformers import pipeline, set_seed
set_seed(0)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
llm = pipeline("text-generation", model=MODEL_ID, device_map="auto",
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)

def run(prompt, system=None, max_new_tokens=300, show=True):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    t0 = time.time()
    out = llm(msgs, max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"][-1]["content"].strip()
    if show:
        print(out)
        print(f"\n[{len(out.split())} words, {time.time()-t0:.0f}s]")
    return out

print("Model:", MODEL_ID, "| device:", llm.device)

## 🌟 Exercise 1: Rewrite and optimize a vague prompt
> Vague prompt: **"Write something about productivity tips."**

### 3 critical issues
1. **No brand, product or goal.** FlowNest is never mentioned, so the output cannot promote the app, and nothing says the goal is downloads or sign-ups. The model writes a generic blog-style text that any company could publish.
2. **No target audience or brand voice.** Who is it for (remote workers? students? managers?) and how should FlowNest sound (expert, warm, playful)? Without this the tone is bland and the tips are too generic to resonate with anyone.
3. **No platform, format or engagement constraints.** LinkedIn rewards a strong hook in the first line, short scannable paragraphs or bullets, a clear call-to-action, a few relevant hashtags and a moderate length. "Something" gives no length, no structure and no CTA, so the result is not ready to publish.

In [ ]:
vague_1 = "Write something about productivity tips."
out_vague_1 = run(vague_1, max_new_tokens=250)

### Improved prompt (role + instructions + format/tone/length constraints + reusable line)

In [ ]:
improved_1 = """Act as a senior B2B social-media copywriter for FlowNest, a focus app that blocks distractions,
schedules deep-work sessions and tracks focus time. Our brand voice is warm, expert and encouraging, never pushy.

Write a LinkedIn post for busy remote professionals and team leads who struggle with constant notifications.
- Start with a one-line hook (a question or a surprising fact) that makes people stop scrolling.
- Give exactly 3 actionable productivity tips, each as a bullet point starting with an emoji and a bold verb.
- Show naturally how FlowNest helps with at least one of the tips (no hard sell).
- End with a clear call-to-action inviting readers to try FlowNest free for 14 days, then 3 relevant hashtags.
- Length: 120-180 words. Short sentences, no jargon.

Use this format whenever we promote productivity tools on social media."""
out_improved_1 = run(improved_1, max_new_tokens=350)

In [ ]:
def check_post(text):
    return {"mentions FlowNest": "flownest" in text.lower(),
            "bullet points": len(re.findall(r"^\s*(?:[-*•]|\d+\.|\S\s*\*\*)", text, flags=re.M)),
            "hashtags": len(re.findall(r"#[A-Za-z]\w+", text)),
            "call-to-action (try/free/download)": bool(re.search(r"\b(try|free|download|sign up|start)\b", text, re.I)),
            "words": len(text.split())}
import pandas as pd
pd.DataFrame({"vague prompt": check_post(out_vague_1), "improved prompt": check_post(out_improved_1)})

ANALYSIS_EX1

## 🌟 Exercise 2: Multi-part prompt for quiz generation
We use a short sample article on volcanic eruptions as the teacher's "uploaded" text.

In [ ]:
article_volcano = """Volcanoes form where molten rock, called magma, rises from deep inside the Earth. Magma collects in a
magma chamber under the volcano. When pressure from gases such as water vapour and carbon dioxide builds up, the magma
pushes through cracks and erupts at the surface, where it is called lava. Some eruptions are explosive and throw ash,
gas and rock fragments high into the sky, like Mount St. Helens in 1980. Others are effusive: lava flows out slowly,
as in Hawaii. Most volcanoes are found along the edges of tectonic plates, especially around the Pacific Ocean in an
area called the Ring of Fire. Scientists called volcanologists monitor earthquakes, gas emissions and ground swelling
to predict eruptions and keep people safe. Volcanic ash can be dangerous, but over time it also makes soil very fertile."""
print(len(article_volcano.split()), "words")

In [ ]:
vague_2 = f"Make a quiz for kids about this article.\n\n{article_volcano}"
out_vague_2 = run(vague_2, max_new_tokens=350)

In [ ]:
improved_2 = f"""You are an experienced 7th-grade science teacher who creates fun, clear classroom materials.

TASK: Using ONLY the article below, create content for a Google Slides lesson for students aged 11-13.

Part 1 - Summary: Summarize the article in exactly 2 simple bullet points (max 20 words each).
Part 2 - Quiz: Write exactly 3 multiple-choice questions about facts stated in the article.
  - Each question has 3 options (A, B, C): 1 correct answer and 2 plausible but clearly wrong distractors.
  - Mark the correct answer on a separate line: "Answer: <letter>".

STYLE: friendly, encouraging tone; simple everyday vocabulary; short sentences; explain any science word you use.

FORMAT (Google Slides ready - one block per slide, no long paragraphs):
Slide 1 - Title: <fun title with one emoji>
Slide 2 - Summary: <2 bullets>
Slide 3 - Question 1 ... Slide 5 - Question 3 (question, options A/B/C, Answer)

ARTICLE:
\"\"\"{article_volcano}\"\"\""""
out_improved_2 = run(improved_2, max_new_tokens=450)

In [ ]:
def check_quiz(text):
    return {"'Slide' blocks": len(re.findall(r"slide\s*\d", text, re.I)),
            "questions (lines ending with ?)": len(re.findall(r"\?\s*$", text, re.M)),
            "options A/B/C": len(re.findall(r"^\s*\(?[ABC][\).:]", text, re.M)),
            "answer keys": len(re.findall(r"answer\s*:", text, re.I)),
            "words": len(text.split())}
pd.DataFrame({"vague prompt": check_quiz(out_vague_2), "improved prompt": check_quiz(out_improved_2)})

ANALYSIS_EX2

## 🌟 Exercise 3: Add context, get better results
A sample monthly finance report stands in for the intern's input.

In [ ]:
finance_report = """Monthly Finance Report - March 2025 (Acme Retail).
Revenue reached $4.2M, up 8% from February ($3.89M) and 12% year-over-year, driven by the spring online campaign;
e-commerce now represents 41% of sales (vs 35% a year ago). Gross margin fell from 46% to 43% because of higher
shipping costs (+18%) and promotional discounts. Operating expenses were $1.35M, 5% over budget, mainly due to two
new hires in customer support and a one-off $60K software migration. EBITDA was $0.46M (11% margin, target 13%).
Cash at month end: $2.1M; inventory days rose from 52 to 61 as spring stock arrived early. Accounts receivable
improved: DSO down from 38 to 33 days. Outlook: Q2 revenue forecast $13M; management plans to renegotiate the
shipping contract in April (expected savings $40K/month) and reduce discounting."""
out_vague_3 = run(f"Summarize this report.\n\n{finance_report}", max_new_tokens=250)

### 1. Missing context

| Context type | Missing? | What should be added? |
|---|---|---|
| **Role** | Yes | "Act as a financial analyst preparing the monthly update", so the model prioritises the numbers that matter (margin, EBITDA, cash) the way an analyst would. |
| **Audience** | Yes | A non-technical executive team: plain language, no accounting jargon, focus on business impact. |
| **Purpose** | Yes | A 3-minute spoken segment of the monthly finance update: highlight performance vs targets, risks and next actions. |
| **Input source** | Partly (the text is pasted, but its nature is not stated) | Say it is the March 2025 monthly report and that only figures from it may be used (no outside data, no invented numbers). |
| **Format/Style** | Yes | Headline, then 3 key takeaways as bullets (each with supporting figures and the change vs last month or target), then 1–2 next steps; confident, neutral tone. |
| **Constraints** | Yes | Max ~250 words (≈ 3 minutes spoken); exactly 3 takeaways; every takeaway includes at least one figure; flag misses vs budget/target explicitly. |

### 2. Rewritten prompt (6 context types)

In [ ]:
improved_3 = f"""Act as a financial analyst at Acme Retail. [ROLE]
You are preparing a 3-minute spoken summary for the monthly finance update meeting. [PURPOSE]
The audience is the non-technical executive team: use plain language and explain what each number means for the business. [AUDIENCE]
Use only the figures in the March 2025 monthly report below; do not add or estimate any number that is not in it. [INPUT SOURCE]

Format: [FORMAT/STYLE]
1. One-sentence headline on overall performance.
2. Exactly 3 key takeaways as bullet points, each with supporting data (value + change vs last month, last year or target).
3. "Next steps": 2 short bullets.
Constraints: maximum 200 words; every takeaway must contain at least one figure; state clearly where we missed budget or target. [CONSTRAINTS]

REPORT:
\"\"\"{finance_report}\"\"\""""
out_improved_3 = run(improved_3, max_new_tokens=350)

In [ ]:
MULT = {"k": 1e3, "thousand": 1e3, "m": 1e6, "million": 1e6}
def numbers(text):
    # normalise "$4.2M", "4.2 million", "$40K", "$40,000" -> 4200000.0 / 40000.0 so that reformatted figures still match
    vals = set()
    for num, unit in re.findall(r"(\d[\d,]*(?:\.\d+)?)\s*(million|thousand|[mk](?![a-z]))?", text, flags=re.I):
        v = float(num.replace(",", ""))
        vals.add(round(v * MULT.get(unit.lower(), 1), 2))
    return vals
report_numbers = numbers(finance_report)
for name, out in [("vague", out_vague_3), ("improved", out_improved_3)]:
    nums = numbers(out)
    print(f"{name:8s}: {len(out.split())} words | {len(nums & report_numbers)} figures from the report | "
          f"{len(nums - report_numbers)} figures NOT in the report: {sorted(nums - report_numbers)[:8]}")

ANALYSIS_EX3

## 🌟 Exercise 4: Match prompt to purpose
**Chosen style: Structured.** It is used for a customer-support chatbot template that handles *"Where is my order?"* and return/refund requests.

In [ ]:
support_system = """You are "ShopBot", the customer support assistant of TrendCart, an online fashion store.
Always answer using EXACTLY this structure:

**1. Summary of your request:** <one sentence restating the customer's issue>
**2. What I found:** <facts from the ORDER DATA only; if information is missing, say "I don't have this information">
**3. Next steps:** <numbered list of max 3 concrete steps the customer or TrendCart will take>
**4. Need more help?** <one line: offer a human agent at support@trendcart.com or chat 9am-6pm>

Rules: be polite and concise (max 120 words), never invent order details, policies or dates,
never ask for card numbers or passwords. Return policy: 30 days from delivery, items unworn, free return label."""

customer_msg = """Hi, I ordered a jacket (order #TC-48213) two weeks ago and it still hasn't arrived. Also, if it doesn't fit, can I return it?
ORDER DATA: order #TC-48213, status: shipped on 2 days ago, carrier: DHL, tracking: DH8812345, estimated delivery: tomorrow."""
out_support = run(customer_msg, system=support_system, max_new_tokens=300)

In [ ]:
sections = ["Summary of your request", "What I found", "Next steps", "Need more help"]
print({s: (s.lower() in out_support.lower()) for s in sections}, "| words:", len(out_support.split()))

ANALYSIS_EX4

## 🌟 Exercise 5: Prompt refinement challenge (style, structure and length)
**PulseOne Mini** constraints: friendly tone, bullet points, max 50 words in total, must mention battery life, fitness tracking and Bluetooth compatibility.

In [ ]:
FEATURES = {"battery life": r"battery", "fitness tracking": r"fitness|track", "Bluetooth compatibility": r"bluetooth"}
FRIENDLY = r"!|you|your|love|meet|say hello|easy|😊|🎉|💪|⌚|🔋"

def evaluate_blurb(text):
    words = len(re.findall(r"[A-Za-z0-9'’-]+", text))
    bullets = [l for l in text.splitlines() if re.match(r"^\s*([-*•]|\d+\.)", l) or re.match(r"^\s*[^\w\s]", l)]
    res = {"words": words, "≤ 50 words": words <= 50, "bullet lines": len(bullets)}
    for name, pat in FEATURES.items():
        res[f"mentions {name}"] = bool(re.search(pat, text, re.I))
    res["friendly markers"] = len(re.findall(FRIENDLY, text, re.I))
    res["PASS"] = res["≤ 50 words"] and all(res[f"mentions {n}"] for n in FEATURES) and res["bullet lines"] >= 3
    return res

In [ ]:
prompt_5_v1 = """Write a friendly product blurb for our new smartwatch, PulseOne Mini, for an email campaign.
Use bullet points, keep it under 50 words, and mention battery life, fitness tracking and Bluetooth compatibility."""
out_5_v1 = run(prompt_5_v1, max_new_tokens=150)
evaluate_blurb(out_5_v1)

In [ ]:
prompt_5_v2 = """You are a friendly email copywriter for a wearable-tech brand.
Write a product blurb for the PulseOne Mini smartwatch.

STRICT RULES - follow all of them:
1. Output ONLY the blurb: no title, no introduction, no explanation, no sign-off.
2. One short opening line (max 8 words), then EXACTLY 3 bullet points starting with "- ".
3. Bullet 1 is about battery life, bullet 2 about fitness tracking, bullet 3 about Bluetooth compatibility.
4. Do not exceed 50 words in total (count every word). Each bullet: max 10 words.
5. Tone: friendly, upbeat, speak directly to the reader ("you"). One emoji per bullet is allowed.
6. Do not invent numbers or specifications."""
out_5_v2 = run(prompt_5_v2, max_new_tokens=120)
evaluate_blurb(out_5_v2)

**Revision v3.** v2 respects the hard constraints, but the model **invented specifications** that nobody gave it ("up to 7 days", "hands-free calls"), despite rule 6. A marketing email must not promise features the product may not have. Rule 6 alone is not enough; the fix is to **provide the facts** and forbid anything else, and to ask explicitly for a warmer, more engaging tone.

In [ ]:
prompt_5_v3 = """You are a friendly, upbeat email copywriter for a wearable-tech brand.
Write a product blurb for the PulseOne Mini smartwatch using ONLY these facts:
- Battery: up to 5 days on a single charge
- Fitness: tracks steps, heart rate, sleep and 20 workout modes
- Connectivity: Bluetooth 5.3, works with iPhone and Android

STRICT RULES:
1. Output ONLY the blurb (no title, no explanation).
2. One opening line of max 8 words that speaks to the reader ("you"), then EXACTLY 3 bullet points starting with "- ".
3. Bullet 1 = battery life, bullet 2 = fitness tracking, bullet 3 = Bluetooth compatibility. Use only the facts above.
4. Do not exceed 50 words in total. Each bullet: max 12 words, starting with one emoji.
5. Friendly, energetic tone: use "you/your" and end the opening line with "!"."""
out_5_v3 = run(prompt_5_v3, max_new_tokens=120)
evaluate_blurb(out_5_v3)

In [ ]:
pd.DataFrame({"prompt v1 (basic)": evaluate_blurb(out_5_v1), "prompt v2 (strict)": evaluate_blurb(out_5_v2),
              "prompt v3 (strict + facts)": evaluate_blurb(out_5_v3)})

ANALYSIS_EX5

## 🌟 Exercise 6: Spotting and mitigating hallucinations
A short sample article stands in for the peer-reviewed paper. It deliberately does **not** contain any "50 % of species extinct by 2050" figure.

In [ ]:
article_marine = """Title: Ocean warming and the redistribution of marine biodiversity (sample abstract).
Sea surface temperatures have risen by about 0.9 °C since 1900. Using 30 years of survey data for 1,200 fish and
invertebrate species, the authors show that 45% of the studied species shifted their range towards the poles, at a
median speed of 59 km per decade. Coral reef habitats declined by 14% between 2009 and 2018, mainly after
marine heatwaves. Local extinctions were recorded at the warm edge of species ranges, but the study does not
estimate global extinction rates. The authors call for marine protected areas that account for future species movements."""

vague_6 = f"Summarize this article about climate change and marine biodiversity, and say how many species will go extinct by 2050.\n\n{article_marine}"
out_vague_6 = run(vague_6, max_new_tokens=250)

In [ ]:
grounded_6 = f"""You are a careful science editor. Summarize the article below for a policy briefing.

Rules:
- Use ONLY information explicitly stated in the article. Do not use outside knowledge.
- Every number you write must appear in the article; copy it exactly.
- After each bullet, quote the supporting sentence from the article in brackets, e.g. [“...”].
- If a question cannot be answered from the article, write exactly: "Not stated in the article."

Output: 3 bullet points, then answer this question: How many marine species are projected to go extinct by 2050?

ARTICLE:
\"\"\"{article_marine}\"\"\""""
out_grounded_6 = run(grounded_6, max_new_tokens=350)

In [ ]:
verify_6 = f"""Fact-check the following claim against the article.
Claim: "Over 50% of marine species are projected to go extinct by 2050."
Answer with one label - SUPPORTED, CONTRADICTED or NOT IN ARTICLE - followed by the exact sentence from the article
that justifies your label (or "none"). Only include claims explicitly stated in the article; if uncertain, say the
information is not available.

ARTICLE:
\"\"\"{article_marine}\"\"\""""
out_verify_6 = run(verify_6, max_new_tokens=120)

In [ ]:
art_nums = numbers(article_marine)
for name, out in [("vague", out_vague_6), ("grounded", out_grounded_6)]:
    extra = sorted(numbers(out) - art_nums)
    print(f"{name:9s}: numbers not in the article: {extra} | says 'not stated': {'not stated' in out.lower()}")

ANALYSIS_EX6